# Roof Collapse Risk — Civil Engineering Analysis

Applies the "Connect to Civil Engineering" framework from the Tephra Hazard Modeling lab
(Assessing the impact of ash deposits using Tephra2 on VICTOR) to this project's three
Mt. Hood communities, using this project's own Tephra2 output instead of the lab's South
Sister walkthrough.

For one representative building per community, this notebook:
1. Converts modeled ash loading (kg/m²) at that community into a roof load (kPa).
2. Compares that load against mean roof-collapse loads by roof class, from
   Reyes-Hardy et al. (2024) — the same source table used on page 7 of the lab.
3. Reports a collapse yes/no call, for both the baseline scenario and the worst-case
   scenario from the parameter sweep (`tephra2_sweep.ipynb` / `tephra2_analysis.ipynb`).

**This notebook does not pick buildings or roof types on its own.** Site selection (one
building in each of Govt. Camp, Rhododendron, and Parkdale), roof area, and roof class
come from a live Google Maps + Street View survey, same as lab Part One — fill those in
under "Site data" below.


## Section 1 — Setup

In [ ]:
import os

import pandas as pd


In [ ]:
GRAVITY = 9.8  # m/s^2

# Points of interest (must match tephra2_analysis.ipynb / tephra2_sweep.ipynb)
poi_names = ["Rhododendron", "Parkdale", "Govt. Camp"]
poi_files = {
    name: f"{name.lower().replace('.', '').replace(' ', '_')}.csv"
    for name in poi_names
}

# Baseline (single representative mid-range eruption) loadings, kg/m^2 -- from the
# isomass/hazard map in CRESCENT_Oral_presentation.docx. Used as a fallback if the
# per-community sweep CSVs (govt_camp.csv, parkdale.csv, rhododendron.csv) from
# tephra2_sweep.ipynb aren't present in this working directory.
BASELINE_MASS_KG_M2 = {
    "Govt. Camp": 43.66,
    "Parkdale": 0.01,
    "Rhododendron": 0.01,
}


## Section 2 — Site data

Fill in from a live Google Maps + Street View survey, one building per community
(lab Part One: pick a building, record its roof area via Google Maps "measure
distance", and assign a roof class from the lab's page-7 table or your own
inspection of construction material).

`roof_class` must be one of `"WE"`, `"MW"`, `"MS"`, `"ST"` (weak → strong, per the
lab's roof classification table).

In [ ]:
BUILDINGS = [
    {
        "site": "Govt. Camp",
        "community": "Govt. Camp",
        "description": "Mt. Hood Cultural Center & Museum",
        "lat": 45.30351444525525,
        "lon": -121.75331979925848,
        "roof_area_m2": 371.81,
        "roof_class": "MW",  # pitched standing-seam metal roof, timber-frame, well maintained
    },
    {
        "site": "Rhododendron",
        "community": "Rhododendron",
        "description": "Zigzag Ranger Station (Zigzag, adjacent to Rhododendron)",
        "lat": 45.34306592961291,
        "lon": -121.94164173037395,
        "roof_area_m2": 862.57,
        "roof_class": "MW",  # pitched shingle roof, timber post-and-beam, well maintained
    },
    {
        "site": "Parkdale",
        "community": "Parkdale",
        "description": "Parkdale Elementary School",
        "lat": 45.522032441102716,
        "lon": -121.59372942243195,
        "roof_area_m2": 3164.11,
        "roof_class": "MS",  # low-pitch composite shingle over reinforced masonry block, institutional construction
    },
]


## Section 3 — Roof collapse thresholds

Mean collapse load (kPa) by roof class, from Reyes-Hardy et al. (2024), Table
"Classifying Roof Types" (Tephra Hazard Modeling lab, page 7).

In [ ]:
ROOF_COLLAPSE_KPA = {
    "WE": {"label": "Weak (metallic/fiber cement/zinc/plastic sheets; old masonry)", "mean_collapse_kpa": 2.0},
    "MW": {"label": "Medium-weak (tiled roof, average timber trusses)", "mean_collapse_kpa": 3.0},
    "MS": {"label": "Medium-strong (flat reinforced concrete, modern pitched)", "mean_collapse_kpa": 4.5},
    "ST": {"label": "Strong (flat concrete slab, recent construction)", "mean_collapse_kpa": 7.0},
}


**Note on the two threshold frameworks in this project:** the Wilson et al. (2014) /
Jenkins et al. (2015) `HAZARD_THRESHOLDS` used elsewhere in this project (1 / 10 / 100 /
1,000 kg/m², see `tephra2_analysis.ipynb`) are categorical hazard labels from volcanology
literature -- "100 kg/m² = roof collapse risk" is a label, not a physics-based load
calculation. It is *not* the same as converting kg/m² to kPa (`mass * gravity`) and
comparing against Reyes-Hardy's roof-class thresholds, which is what this notebook (and
the lab) does. The two disagree by roughly a factor of 200-700: 100 kg/m² converts to
only ~0.98 kPa, below even the weakest (WE) roof class's 2.0 kPa mean collapse load. A
roof needs on the order of 200-700 kg/m² (depending on roof class) before the physics-based
calculation calls a collapse. Cite whichever framework you're actually using in the
writeup -- don't quote a Wilson/Jenkins "roof collapse risk" label next to a Reyes-Hardy
collapse call as if they're measuring the same thing.

## Section 4 — Community ash loading (baseline + worst case)

In [ ]:
def community_loadings(community):
    """Return (baseline_kg_m2, worst_case_kg_m2, source) for one community.

    Prefers the actual sweep output (govt_camp.csv / parkdale.csv / rhododendron.csv,
    written by tephra2_sweep.ipynb) if present in this directory; falls back to the
    baseline figure already reported in the presentation, with worst case left as
    unknown.
    """
    fname = poi_files[community]
    if os.path.exists(fname):
        df = pd.read_csv(fname)
        df.loc[df["mass_kg_m2"] < 1e-300, "mass_kg_m2"] = 0.0
        worst_case = df["mass_kg_m2"].max()
        baseline = BASELINE_MASS_KG_M2.get(community)
        return baseline, worst_case, fname
    return BASELINE_MASS_KG_M2.get(community), None, "baseline figure in presentation (no sweep CSV found)"


for name in poi_names:
    baseline, worst_case, source = community_loadings(name)
    wc = f"{worst_case:.3g} kg/m^2" if worst_case is not None else "unavailable"
    print(f"{name}: baseline={baseline} kg/m^2, worst_case={wc}  (source: {source})")


## Section 5 — Load and collapse calculation

In [ ]:
def deposit_load_kpa(mass_kg_m2):
    """Deposit weight per area, in kPa, from mass per area (kg/m^2) and gravity."""
    return mass_kg_m2 * GRAVITY / 1000


def check_collapse(mass_kg_m2, roof_class):
    threshold = ROOF_COLLAPSE_KPA[roof_class]["mean_collapse_kpa"]
    load = deposit_load_kpa(mass_kg_m2)
    return {
        "load_kpa": load,
        "threshold_kpa": threshold,
        "collapse": load >= threshold,
    }


## Section 6 — Results

Requires `roof_area_m2` and `roof_class` to be filled in for each building in
`BUILDINGS` above. `roof_area_m2` isn't used in the collapse call itself (load is
per unit area) but is carried through for reference against the lab's data table
and for estimating total roof-load force if needed later.

In [ ]:
rows = []
for b in BUILDINGS:
    if b["roof_class"] is None:
        print(f"Skipping {b['site']}: roof_class not filled in yet.")
        continue

    baseline, worst_case, _ = community_loadings(b["community"])

    for scenario, mass in [("baseline", baseline), ("worst case", worst_case)]:
        if mass is None:
            continue
        result = check_collapse(mass, b["roof_class"])
        rows.append({
            "site": b["site"],
            "description": b["description"],
            "roof_area_m2": b["roof_area_m2"],
            "roof_class": b["roof_class"],
            "scenario": scenario,
            "mass_kg_m2": mass,
            "load_kpa": round(result["load_kpa"], 3),
            "threshold_kpa": result["threshold_kpa"],
            "collapse": "YES" if result["collapse"] else "NO",
        })

results = pd.DataFrame(rows)
results


## Limitations

- Community-level modeled loading is used as a proxy for the load at each specific
  building's coordinates — Tephra2 wasn't re-run at the exact building point, so
  this assumes loading is roughly uniform within a community's footprint (a
  simplification also made in `infrastructure_exposure.ipynb`'s 5 km buffers).
- The Reyes-Hardy et al. (2024) thresholds are mean collapse loads from a general
  roof survey (Tajogaite eruption, La Palma), not calibrated specifically for
  Cascade-region construction.
- Roof area and roof class are visual-survey judgment calls (Street View imagery),
  not structural inspection — treat the collapse call as a screening-level estimate,
  consistent with how the lab itself frames it.